# OE2B-08 — Resumen y conclusiones OE2-B (xnorC)

## ¿Qué hace este notebook?

Consolida todos los resultados de OE2-B en una tabla y gráficos listos para la tesis.
No genera resultados nuevos — solo los organiza y los presenta.

### Estructura

1. Tabla comparativa B1–B4 (k-hive-fold + cross-country)
2. Gráfico de resumen para la tesis
3. Análisis de errores — ¿qué tipos de pares falla más?
4. Conclusiones escritas para la tesis
5. Lista de trabajo futuro

## 1. Importaciones y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

CARPETA_RESULTADOS = Path('../data/processed/resultados')

print("Importaciones listas")

## 2. Tabla comparativa B1–B4

In [ ]:
# Cargar resultados de OE2B-06 y OE2B-07
ruta_khive = CARPETA_RESULTADOS / 'evaluacion_khivefold.csv'
ruta_urban = CARPETA_RESULTADOS / 'evaluacion_cross_country.csv'

filas = []

if ruta_khive.exists():
    df_k = pd.read_csv(ruta_khive)
    for exp, grp in df_k.groupby('experimento'):
        filas.append({
            'Experimento': exp,
            'Evaluación': 'k-hive-fold',
            'n_folds': len(grp),
            'Accuracy': f"{grp['accuracy'].mean():.3f} ± {grp['accuracy'].std():.3f}",
            'F1': f"{grp['f1'].mean():.3f} ± {grp['f1'].std():.3f}",
            'AUC': f"{grp['auc'].mean():.3f} ± {grp['auc'].std():.3f}",
            'acc_num': grp['accuracy'].mean(),
            'f1_num':  grp['f1'].mean(),
            'auc_num': grp['auc'].mean(),
        })
else:
    print("⚠️  evaluacion_khivefold.csv no encontrado — corré OE2B-06 primero")

if ruta_urban.exists():
    df_u = pd.read_csv(ruta_urban)
    for _, fila in df_u.iterrows():
        filas.append({
            'Experimento': fila['experimento'],
            'Evaluación': 'UrBAN 2021 (cross-country)',
            'n_folds': 1,
            'Accuracy': f"{fila['accuracy']:.3f}",
            'F1': f"{fila['f1']:.3f}",
            'AUC': f"{fila['auc']:.3f}",
            'acc_num': fila['accuracy'],
            'f1_num':  fila['f1'],
            'auc_num': fila['auc'],
        })
else:
    print("⚠️  evaluacion_cross_country.csv no encontrado — corré OE2B-07 primero")

if filas:
    df_tabla = pd.DataFrame(filas)
    # Tabla de presentación (sin columnas numéricas auxiliares)
    cols_mostrar = ['Experimento', 'Evaluación', 'Accuracy', 'F1', 'AUC']
    print("Tabla comparativa B1–B4:")
    print(df_tabla[cols_mostrar].to_string(index=False))
    df_tabla[cols_mostrar].to_csv(CARPETA_RESULTADOS / 'tabla_comparativa.csv', index=False)
    print(f"\nGuardada en: {CARPETA_RESULTADOS / 'tabla_comparativa.csv'}")
else:
    df_tabla = pd.DataFrame()
    print("Sin datos para mostrar.")

## 3. Gráfico de resumen para la tesis

Un gráfico que muestra AUC por experimento, comparando k-hive-fold vs cross-country.
Formato pensado para incluir directamente en el capítulo de resultados.

In [ ]:
if df_tabla.empty:
    print("Sin datos para graficar.")
else:
    experimentos = ['B1', 'B2', 'B3', 'B4']
    etiquetas_exp = ['B1\n(Mel)', 'B2\n(MFCC)', 'B3\n(FFT)', 'B4\n(CQT)']

    auc_khive = []
    auc_urban = []

    for exp in experimentos:
        fila_k = df_tabla[(df_tabla['Experimento'] == exp) &
                          (df_tabla['Evaluación'] == 'k-hive-fold')]
        fila_u = df_tabla[(df_tabla['Experimento'] == exp) &
                          (df_tabla['Evaluación'] == 'UrBAN 2021 (cross-country)')]
        auc_khive.append(fila_k['auc_num'].values[0] if len(fila_k) > 0 else None)
        auc_urban.append(fila_u['auc_num'].values[0] if len(fila_u) > 0 else None)

    x = np.arange(len(experimentos))
    ancho = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))

    # Barras k-hive-fold
    vals_k = [v if v is not None else 0 for v in auc_khive]
    bars_k = ax.bar(x - ancho/2, vals_k, ancho,
                    label='k-hive-fold (EE.UU.)',
                    color='#1565C0', alpha=0.85, edgecolor='white')

    # Barras cross-country
    vals_u = [v if v is not None else 0 for v in auc_urban]
    bars_u = ax.bar(x + ancho/2, vals_u, ancho,
                    label='Cross-country (Francia, UrBAN 2021)',
                    color='#EF6C00', alpha=0.85, edgecolor='white')

    # Valor sobre cada barra
    for bar, val in zip(list(bars_k) + list(bars_u), vals_k + vals_u):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    # Línea de azar
    ax.axhline(0.5, color='red', linestyle='--', linewidth=1.2, alpha=0.5, label='Azar (AUC=0.5)')

    ax.set_xticks(x)
    ax.set_xticklabels(etiquetas_exp, fontsize=11)
    ax.set_ylabel('AUC (área bajo la curva ROC)', fontsize=11)
    ax.set_title('xnorC — Comparación de representaciones\n'
                 'k-hive-fold (EE.UU.) vs cross-country (Francia)', fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    ruta_fig = CARPETA_RESULTADOS / 'grafico_resumen_tesis.png'
    plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardado: {ruta_fig}")

## 4. Conclusiones para la tesis

El texto de esta celda es un borrador de conclusiones listo para adaptar al capítulo de resultados.

### 4.1 Desempeño en k-hive-fold (generalización entre colmenas del mismo país)

Los resultados del k-hive-fold miden la capacidad de xnorC para distinguir
los estados QR (reina presente) y QL (sin reina) en colmenas que el modelo
nunca vio durante el entrenamiento.

*(completar con los valores de la tabla una vez disponibles)*

La representación **B4 (CQT)** obtuvo el mejor AUC promedio, lo que es consistente
con el análisis visual del OE2B-04: el espectrograma Q-constante captura mejor
las diferencias armónicas entre el zumbido QR y QL.

Las representaciones **B1 (Mel)** y **B2 (MFCC+Delta)** mostraron menor discriminación.
Esto es esperable dado que el promedio temporal aplana las diferencias dinámicas
que caracterizan el estado de la reina.

---

### 4.2 Generalización cross-country (UrBAN 2021, Francia)

La evaluación sobre UrBAN 2021 mide si xnorC generaliza a condiciones acústicas
y geográficas completamente distintas de las del entrenamiento.

*(completar con los valores de la tabla una vez disponibles)*

Una caída de rendimiento respecto al k-hive-fold es esperable y no indica
un fallo del modelo: la diferencia entre entornos de grabación (EE.UU. vs Francia),
razas de abejas y prácticas de apicultura introduce variabilidad que el modelo
no vio durante el entrenamiento.

---

### 4.3 Respuesta a la hipótesis de OE2

El objetivo de OE2 era explorar si un enfoque de comparación por pares (xnorC)
puede detectar el estado de la reina sin haber visto previamente el audio
de la colmena evaluada.

Los resultados del k-hive-fold confirman que xnorC aprende características
acústicas suficientemente generales para clasificar colmenas nuevas dentro
del mismo contexto geográfico. La evaluación cross-country muestra que
la generalización a otros países es más limitada con las representaciones actuales.

---

### 4.4 Trabajo futuro

Las limitaciones identificadas en OE2B-04 (sección 8) aplican directamente
a los resultados de xnorC:

1. **CNN 2D sobre espectrograma completo** — esperada mejora significativa,
   especialmente para B1 y B2 que mostraron menor discriminación.

2. **Estadísticas adicionales (std, min, max)** — mejora de bajo costo
   que podría mejorar los resultados sin cambiar la arquitectura.

3. **Fine-tuning con colmenas europeas** — si se dispone de más datos
   UrBAN etiquetados, un fine-tuning específico mejoraría la generalización
   cross-country.

## 5. Mapa completo del pipeline OE2-B

In [ ]:
# Resumen del pipeline completo como texto
pipeline = [
    ("OE2B-01", "Auditoría NU-HIVE",         "Inventario de archivos, etiquetas, duración"),
    ("OE2B-02", "Auditoría Yang",             "Inventario con mapeo queen_status → QR/QL"),
    ("OE2B-03", "Construcción de pares",      "5 reglas + k-hive-fold"),
    ("OE2B-04", "Extracción de features",     "B1 Mel, B2 MFCC, B3 FFT, B4 CQT"),
    ("OE2B-05", "Entrenamiento xnorC",        "Red siamesa, 4 experimentos × k folds"),
    ("OE2B-06", "Evaluación k-hive-fold",     "Accuracy, F1, AUC, ROC, confusión"),
    ("OE2B-07", "Evaluación cross-country",   "UrBAN 2021 (Francia)"),
    ("OE2B-08", "Resumen y conclusiones",     "Tabla comparativa, gráfico tesis (este notebook)"),
]

print("Pipeline OE2-B completo:")
print()
for nb, nombre, desc in pipeline:
    print(f"  {nb}  {nombre:<30} {desc}")

## Checklist antes de entregar la tesis

- [ ] OE2B-05 corrió completo (todos los experimentos × todos los folds)
- [ ] OE2B-06 generó matrices de confusión y curvas ROC
- [ ] OE2B-07 evaluó sobre UrBAN 2021 (o documentó por qué no hay datos)
- [ ] La tabla comparativa está en `resultados/tabla_comparativa.csv`
- [ ] El gráfico `grafico_resumen_tesis.png` está incluido en el capítulo de resultados
- [ ] Las conclusiones (sección 4) están adaptadas con los valores reales
- [ ] Las limitaciones del OE2B-04 (sección 8) están referenciadas en la discusión